<a href="https://colab.research.google.com/github/nisabulut/digital-image-processing/blob/main/10_3_goruntu_isleme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%%bash
# OpenCV geliştirme paketlerini yükle
apt-get update -qq && apt-get install -y -qq libopencv-dev

cat << 'EOF' > stereo_metrics.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
#include <limits>
#include <cmath>
#include <string>

// Ortak fonksiyon pointer tipi:
// (solGoruntu, sagGoruntu, solX, ortaY, sagX, yaricap) -> skor
using BlockCostFunc = double (*)(const cv::Mat&, const cv::Mat&, int, int, int, int);

// 1. SAD: Düşük değer = daha iyi eşleşme
double metricSAD(const cv::Mat& left, const cv::Mat& right, int x, int y, int rx, int r) {
    double sad = 0.0;
    for (int u = -r; u <= r; ++u) {
        const uchar* pL = left.ptr<uchar>(y + u);
        const uchar* pR = right.ptr<uchar>(y + u);
        for (int v = -r; v <= r; ++v) {
            sad += std::abs(static_cast<double>(pL[x + v]) - static_cast<double>(pR[rx + v]));
        }
    }
    return sad;
}

// 2. SSD: Düşük değer = daha iyi eşleşme
double metricSSD(const cv::Mat& left, const cv::Mat& right, int x, int y, int rx, int r) {
    double ssd = 0.0;
    for (int u = -r; u <= r; ++u) {
        const uchar* pL = left.ptr<uchar>(y + u);
        const uchar* pR = right.ptr<uchar>(y + u);
        for (int v = -r; v <= r; ++v) {
            double diff = static_cast<double>(pL[x + v]) - static_cast<double>(pR[rx + v]);
            ssd += diff * diff;
        }
    }
    return ssd;
}

// 3. ZNCC: Yüksek değer (1.0'a yakın) = daha iyi eşleşme
double metricZNCC(const cv::Mat& left, const cv::Mat& right, int x, int y, int rx, int r) {
    int n = (2 * r + 1) * (2 * r + 1);
    double sumL = 0.0, sumR = 0.0;

    for (int u = -r; u <= r; ++u) {
        const uchar* pL = left.ptr<uchar>(y + u);
        const uchar* pR = right.ptr<uchar>(y + u);
        for (int v = -r; v <= r; ++v) {
            sumL += static_cast<double>(pL[x + v]);
            sumR += static_cast<double>(pR[rx + v]);
        }
    }

    double meanL = sumL / n;
    double meanR = sumR / n;

    double num = 0.0, denL = 0.0, denR = 0.0;
    for (int u = -r; u <= r; ++u) {
        const uchar* pL = left.ptr<uchar>(y + u);
        const uchar* pR = right.ptr<uchar>(y + u);
        for (int v = -r; v <= r; ++v) {
            double dL = static_cast<double>(pL[x + v]) - meanL;
            double dR = static_cast<double>(pR[rx + v]) - meanR;
            num  += dL * dR;
            denL += dL * dL;
            denR += dR * dR;
        }
    }

    double denom = std::sqrt(denL * denR);
    if (denom < 1e-9) {
        return -1.0; // Düz/kontrastsız bloklarda geçersiz korelasyon
    }
    return num / denom;
}

cv::Mat computeDisparityWithMetric(const cv::Mat& left,
                                   const cv::Mat& right,
                                   int numDisparities,
                                   int blockSize,
                                   BlockCostFunc scoreFunc,
                                   bool higherIsBetter) {
    CV_Assert(scoreFunc != nullptr);
    int rows = left.rows;
    int cols = left.cols;
    int r = blockSize / 2;

    cv::Mat disp = cv::Mat::zeros(rows, cols, CV_32F);
    int startX = std::max(r, r + numDisparities - 1);
    int endX   = cols - r;

    for (int y = r; y < rows - r; ++y) {
        for (int x = startX; x < endX; ++x) {
            double bestScore = higherIsBetter ? -std::numeric_limits<double>::infinity()
                                              :  std::numeric_limits<double>::infinity();
            int bestD = 0;

            for (int d = 0; d < numDisparities; ++d) {
                int rx = x - d;
                if (rx - r < 0) break;

                double score = scoreFunc(left, right, x, y, rx, r);
                bool isBetter = higherIsBetter ? (score > bestScore) : (score < bestScore);

                if (isBetter) {
                    bestScore = score;
                    bestD = d;
                }
            }

            disp.at<float>(y, x) = static_cast<float>(bestD);
        }
    }

    return disp;
}

void saveColoredMap(const cv::Mat& dispFloat, const std::string& filename) {
    double minVal, maxVal;
    cv::minMaxLoc(dispFloat, &minVal, &maxVal);

    cv::Mat dispNorm, dispColor;
    double scale = (maxVal - minVal > 0.0) ? (255.0 / (maxVal - minVal)) : 1.0;
    dispFloat.convertTo(dispNorm, CV_8U, scale, -minVal * scale);
    cv::applyColorMap(dispNorm, dispColor, cv::COLORMAP_TURBO);
    cv::imwrite(filename, dispColor);
}

int main() {
    cv::Mat imgLeft  = cv::imread("scene1.row3.col1.ppm", cv::IMREAD_GRAYSCALE);
    cv::Mat imgRight = cv::imread("scene1.row3.col3.ppm", cv::IMREAD_GRAYSCALE);

    if (imgLeft.empty() || imgRight.empty()) {
        std::cerr << "Goruntuler yuklenemedi!" << std::endl;
        return -1;
    }

    int numDisparities = 128;
    int blockSize = 9;

    struct TestCase {
        const char* name;
        BlockCostFunc fn;
        bool higherIsBetter;
        const char* outFile;
    };

    TestCase tests[] = {
        {"SAD",  &metricSAD,  false, "disp_sad.png"},
        {"SSD",  &metricSSD,  false, "disp_ssd.png"},
        {"ZNCC", &metricZNCC, true,  "disp_zncc.png"}
    };

    for (const auto& t : tests) {
        int64 t0 = cv::getTickCount();
        cv::Mat disp = computeDisparityWithMetric(imgLeft, imgRight, numDisparities, blockSize, t.fn, t.higherIsBetter);
        double elapsedMs = (cv::getTickCount() - t0) * 1000.0 / cv::getTickFrequency();

        saveColoredMap(disp, t.outFile);
        std::cout << t.name << " tamamlandi (" << elapsedMs << " ms) -> " << t.outFile << std::endl;
    }

    return 0;
}
EOF

g++ -O3 stereo_metrics.cpp -o stereo_metrics `pkg-config --cflags --libs opencv4`
./stereo_metrics

SAD tamamlandi (1224.04 ms) -> disp_sad.png
SSD tamamlandi (1145.4 ms) -> disp_ssd.png
ZNCC tamamlandi (2718.26 ms) -> disp_zncc.png


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
